In [47]:
# Importations
import os
import psycopg
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import round,col, dayofmonth,month,year,quarter,when 
from pyspark.sql import functions as F

In [48]:
# Chargement des variables d'environnement (.env)
load_dotenv()


True

In [49]:
# Paramètres de connexion et chemins
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

In [50]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WindPowerSilverTransformation") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

In [51]:
# Récupère la table windturbinepower_raw depuis PostgreSQL dans un DataFrame Spark
windturbinepower_raw_df = spark.read.jdbc(
    url=JDBC_URL,
    table='bronze.windturbinepower_raw',
    properties=JDBC_PROPS
)

In [52]:
# Nettoyage et enrichissement des données
# Transformation: arrondir les colonnes "energy_produced" et "wind_speed"
# Transformation: créer la colonne "day" à partir de "date" avec la méthode "dayofmonth"
# Transformation: créer la colonne "month" à partir de "date" avec la méthode "month"
# Transformation: créer la colonne "quarter" à partir de "date" avec la méthode "quarter"
# Transformation: créer la colonne "year" à partir de "date" avec la méthode "year"
# Transformation: formater la colonne "time" au format "HH:mm:ss"
# Transformation: créer les colonnes "hour_of_day", "minute_of_hour", "second_of_minute" à partir de "time"

df_transformed = (
    windturbinepower_raw_df
    .withColumn("wind_speed", round(col("wind_speed"), 2))
    .withColumn("energy_produced", round(col("energy_produced"), 2))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("year", year(col("date")))
    .withColumn("time", F.date_format(col("time"), "HH:mm:ss"))
    .withColumn("hour_of_day", F.hour(col("time")))
    .withColumn("minute_of_hour", F.minute(col("time")))
    .withColumn("second_of_minute", F.second(col("time")))
    .withColumn(
        "time_period",
        when((col("hour_of_day") >= 5) & (col("hour_of_day") < 12), "Morning")
        .when((col("hour_of_day") >= 12) & (col("hour_of_day") < 17), "Afternoon")
        .when((col("hour_of_day") >= 17) & (col("hour_of_day") < 21), "Evening")
        .otherwise("Night")
    )
)

# Suppression des colonnes 
df_transformed = df_transformed.drop("ingested_at", "source_file", "batch_id","measured_at")

df_transformed.show(5)

+-------------+----------+--------+------------+--------+-------------+---------+-----------+--------+----------+----------------------+----------+--------------+---------------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|production_id|      date|    time|turbine_name|capacity|location_name| latitude|  longitude|  region|    status|responsible_department|wind_speed|wind_direction|energy_produced|day|month|quarter|year|hour_of_day|minute_of_hour|second_of_minute|time_period|
+-------------+----------+--------+------------+--------+-------------+---------+-----------+--------+----------+----------------------+----------+--------------+---------------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|         6049|2024-06-15|00:00:00|   Turbine A|    2200|   Location 1|34.052200|-118.243700|Region A|    Online|            Operations|      5.74|            SW|        1783.39| 15|    6|      2|2024|          0|             

In [53]:
# Définition de la table silver.windpowerturbine_cleaned avec les types de données adaptés BI
create_table_sql = """
CREATE TABLE IF NOT EXISTS silver.windpowerturbine_cleaned (
    production_id INTEGER,
    date DATE,
    time TIME,
    turbine_name VARCHAR(100),
    capacity INTEGER,
    location_name VARCHAR(100),
    latitude NUMERIC(9,6),
    longitude NUMERIC(9,6),
    region VARCHAR(100),
    status VARCHAR(50),
    responsible_department VARCHAR(100),
    wind_speed NUMERIC(6,2),
    wind_direction VARCHAR(50),
    energy_produced NUMERIC(12,2),
    day INTEGER,
    month INTEGER,
    quarter INTEGER,
    year INTEGER,
    hour_of_day INTEGER,
    minute_of_hour INTEGER,
    second_of_minute INTEGER,
    time_period VARCHAR(20)
);
"""

# Connexion à PostgreSQL et exécution du script de création de table
with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    autocommit=True
) as conn:
    with conn.cursor() as cur:
        # Crée le schéma s'il n'existe pas
        cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
        
        # Crée la table avec script SQL
        cur.execute(create_table_sql)
        print("Schéma silver et table windpowerturbine_cleaned créés avec les types corrects")
        
        # Ajoute la clé primaire après création de la table
        try:
            cur.execute("""
                ALTER TABLE silver.windpowerturbine_cleaned
                ADD CONSTRAINT windpowerturbine_cleaned_pkey PRIMARY KEY (production_id);
            """)
            print("Clé primaire ajoutée sur (production_id)")
        except (psycopg.errors.DuplicateTable, psycopg.errors.DuplicateObject, psycopg.errors.InvalidTableDefinition):
            print("Clé primaire existe déjà")
        

Schéma silver et table windpowerturbine_cleaned créés avec les types corrects
Clé primaire existe déjà


In [ ]:
# Insertion des données dans la table en mode UPSERT
total_count = df_transformed.count()
print(f"Lignes à traiter: {total_count}")

# Conversion du DataFrame Spark en Pandas pour l'upsert
df_transformed_pd = df_transformed.toPandas()

upsert_sql = """
    INSERT INTO silver.windpowerturbine_cleaned (
        production_id, date, time, turbine_name, capacity, location_name, latitude, longitude, region, status,responsible_department, wind_speed, wind_direction, energy_produced, day, month, quarter, year,hour_of_day, minute_of_hour, second_of_minute, time_period
        ) 
    VALUES (
        %(production_id)s, %(date)s, %(time)s, %(turbine_name)s, %(capacity)s, %(location_name)s,
        %(latitude)s, %(longitude)s, %(region)s, %(status)s, %(responsible_department)s,
        %(wind_speed)s, %(wind_direction)s, %(energy_produced)s, %(day)s, %(month)s,
        %(quarter)s, %(year)s, %(hour_of_day)s, %(minute_of_hour)s,
        %(second_of_minute)s, %(time_period)s
        )
    ON CONFLICT (production_id) DO UPDATE SET
        date = EXCLUDED.date,
        time = EXCLUDED.time,
        turbine_name = EXCLUDED.turbine_name,
        capacity = EXCLUDED.capacity,
        location_name = EXCLUDED.location_name,
        latitude = EXCLUDED.latitude,
        longitude = EXCLUDED.longitude,
        region = EXCLUDED.region,
        status = EXCLUDED.status,
        responsible_department = EXCLUDED.responsible_department,
        wind_speed = EXCLUDED.wind_speed,
        wind_direction = EXCLUDED.wind_direction,
        energy_produced = EXCLUDED.energy_produced,
        day = EXCLUDED.day,
        month = EXCLUDED.month,
        quarter = EXCLUDED.quarter,
        year = EXCLUDED.year,
        hour_of_day = EXCLUDED.hour_of_day,
        minute_of_hour = EXCLUDED.minute_of_hour,
        second_of_minute = EXCLUDED.second_of_minute,
        time_period = EXCLUDED.time_period
    RETURNING (xmax = 0) AS inserted;
    """

new_rows_count = 0
duplicates_avoided_count = 0

with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    autocommit=True
) as conn:
    with conn.cursor() as cur:
        for _, row in df_transformed_pd.iterrows():
            params = {str(k): v for k, v in row.items()}
            cur.execute(upsert_sql, params)
            result = cur.fetchone()
            inserted = bool(result[0]) if result is not None else False
            if inserted:
                new_rows_count += 1
            else:
                duplicates_avoided_count += 1
        
print(f"{total_count} lignes traitées avec UPSERT")
print(f"Nouvelles lignes: {new_rows_count}")
print(f"Doublons évités: {duplicates_avoided_count}")

Lignes à traiter: 432
432 lignes traitées avec UPSERT
Nouvelles lignes: 0
Doublons évités: 432


In [55]:
# Arrêt de la session Spark
spark.stop()